In [2]:
%pip install -r requirements.txt

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.0-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.18.0-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached numpy-2.5.0-cp314-cp314-win_amd64.whl (12.6 MB)
Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl (8.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.18.0-cp314-cp314-win_amd64.whl (37.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   ---------------------------------------- 0/8 [tzdata]
   -------------------------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_validate, KFold
from sklearn.model_selection import GridSearchCV

import json
from pathlib import Path

In [4]:
df = pd.read_csv("data/cardata.csv")

df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [5]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Dataset Shape: (301, 9)

Column Names:
['Car_Name', 'Year', 'Selling_Price', 'Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner']

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    str    
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Kms_Driven     301 non-null    int64  
 5   Fuel_Type      301 non-null    str    
 6   Seller_Type    301 non-null    str    
 7   Transmission   301 non-null    str    
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 21.3 KB

Missing Values:
Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Kms_Driven       0
Fuel_Type        0
Seller_Type      0
Transmission     0
Owner       

In [6]:
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset Shape After Removing Duplicates:", df.shape)

Dataset Shape After Removing Duplicates: (299, 9)


In [7]:
REFERENCE_YEAR = 2019

df["Car_Age"] = REFERENCE_YEAR - df["Year"]

df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Car_Age
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0,5
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0,6
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0,2
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0,8
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0,5


In [8]:
df[["Year", "Car_Age"]].head(10)

,Year,Car_Age
0,2014,5
1,2013,6
2,2017,2
3,2011,8
4,2014,5
5,2018,1
6,2015,4
7,2015,4
8,2016,3
9,2015,4


In [9]:
features = [
    "Present_Price",
    "Kms_Driven",
    "Fuel_Type",
    "Seller_Type",
    "Transmission",
    "Owner",
    "Car_Age"
]

target = "Selling_Price"

X = df[features]
y = df[target]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

X.head()

Features Shape: (299, 7)
Target Shape: (299,)


,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Car_Age
0,5.59,27000,Petrol,Dealer,Manual,0,5
1,9.54,43000,Diesel,Dealer,Manual,0,6
2,9.85,6900,Petrol,Dealer,Manual,0,2
3,4.15,5200,Petrol,Dealer,Manual,0,8
4,6.87,42450,Diesel,Dealer,Manual,0,5


In [10]:
numerical_features = [
    "Present_Price",
    "Kms_Driven",
    "Owner",
    "Car_Age"
]

categorical_features = [
    "Fuel_Type",
    "Seller_Type",
    "Transmission"
]

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

Numerical Features: ['Present_Price', 'Kms_Driven', 'Owner', 'Car_Age']
Categorical Features: ['Fuel_Type', 'Seller_Type', 'Transmission']


In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (239, 7)
Testing Features: (60, 7)
Training Target: (239,)
Testing Target: (60,)


In [13]:
models = {
    "Linear Regression": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LinearRegression())
        ]
    ),

    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=200,
                    random_state=42
                )
            )
        ]
    ),

    "Gradient Boosting": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                GradientBoostingRegressor(
                    n_estimators=200,
                    random_state=42
                )
            )
        ]
    )
}

print("Models Created:")

for model_name in models:
    print("-", model_name)

Models Created:
- Linear Regression
- Random Forest
- Gradient Boosting


In [14]:
results = []

for model_name, pipeline in models.items():
    print(f"Training {model_name}...")

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

    print(f"{model_name} training completed.\n")

Training Linear Regression...
Linear Regression training completed.

Training Random Forest...
Random Forest training completed.

Training Gradient Boosting...
Gradient Boosting training completed.



In [15]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
).reset_index(drop=True)

results_df

,Model,MAE,RMSE,R2 Score
0,Gradient Boosting,1.152016,2.462328,0.764754
1,Linear Regression,1.472892,2.524035,0.752815
2,Random Forest,1.538803,3.740433,0.457157


In [16]:
best_model_name = results_df.iloc[0]["Model"]

best_model = models[best_model_name]

print("Best Model:", best_model_name)

Best Model: Gradient Boosting


In [17]:
best_predictions = best_model.predict(X_test)

comparison_df = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": best_predictions
})

comparison_df["Difference"] = (
    comparison_df["Actual Price"]
    - comparison_df["Predicted Price"]
).abs()

comparison_df.head(10)

,Actual Price,Predicted Price,Difference
0,8.99,9.504382,0.514382
1,8.35,8.790084,0.440084
2,0.45,0.479135,0.029135
3,7.45,7.161053,0.288947
4,5.25,14.109354,8.859354
5,5.25,4.741860,0.508140
6,5.85,5.128410,0.721590
7,1.15,1.312570,0.162570
8,9.25,8.109005,1.140995
9,0.38,0.315852,0.064148


In [18]:
comparison_df = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": best_predictions
})

comparison_df["Difference"] = (
    comparison_df["Actual Price"]
    - comparison_df["Predicted Price"]
).abs()

comparison_df = comparison_df.sort_values(
    by="Difference",
    ascending=False
).reset_index(drop=True)

comparison_df.head(10)

,Actual Price,Predicted Price,Difference
0,33.00,23.290938,9.709062
1,2.50,11.833653,9.333653
2,5.25,14.109354,8.859354
3,4.00,11.380545,7.380545
4,12.50,8.933020,3.566980
5,12.90,10.938733,1.961267
6,11.75,13.665978,1.915978
7,4.90,6.723670,1.823670
8,6.25,8.044278,1.794278
9,8.25,10.000499,1.750499


In [19]:
error_analysis_df = X_test.copy()

error_analysis_df["Actual_Price"] = y_test
error_analysis_df["Predicted_Price"] = best_predictions

error_analysis_df["Difference"] = (
    error_analysis_df["Actual_Price"]
    - error_analysis_df["Predicted_Price"]
).abs()

error_analysis_df = error_analysis_df.sort_values(
    by="Difference",
    ascending=False
)

error_analysis_df.head(10)

,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Car_Age,Actual_Price,Predicted_Price,Difference
63,36.23,6000,Diesel,Dealer,Automatic,0,2,33.00,23.290938,9.709062
84,23.73,142000,Petrol,Individual,Automatic,3,13,2.50,11.833653,9.333653
77,22.83,80000,Petrol,Dealer,Automatic,0,9,5.25,14.109354,8.859354
92,22.78,89000,Petrol,Dealer,Automatic,0,11,4.00,11.380545,7.380545
82,13.46,38000,Diesel,Dealer,Manual,0,4,12.50,8.933020,3.566980
248,13.60,35934,Diesel,Dealer,Manual,0,3,12.90,10.938733,1.961267
209,14.79,43535,Diesel,Dealer,Manual,0,4,11.75,13.665978,1.915978
73,8.93,83000,Diesel,Dealer,Manual,0,5,4.90,6.723670,1.823670
277,13.60,40126,Petrol,Dealer,Manual,0,5,6.25,8.044278,1.794278
280,14.00,63000,Diesel,Dealer,Manual,0,5,8.25,10.000499,1.750499


In [20]:
df["Selling_Price"].describe()

count    299.000000
mean       4.589632
std        4.984240
min        0.100000
25%        0.850000
50%        3.510000
75%        6.000000
max       35.000000
Name: Selling_Price, dtype: float64

In [21]:
print("Cars with Selling Price above 20 Lakhs:")

df[df["Selling_Price"] > 20][
    [
        "Car_Name",
        "Selling_Price",
        "Present_Price",
        "Year",
        "Kms_Driven"
    ]
].sort_values(
    by="Selling_Price",
    ascending=False
)

Cars with Selling Price above 20 Lakhs:


,Car_Name,Selling_Price,Present_Price,Year,Kms_Driven
85,land cruiser,35.00,92.60,2010,78000
63,fortuner,33.00,36.23,2017,6000
62,fortuner,23.50,35.96,2015,47000
50,fortuner,23.00,30.61,2015,40000
81,innova,23.00,25.39,2017,15000
94,innova,20.75,25.39,2016,29000


In [23]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Cross-validation configured with 5 folds.")

Cross-validation configured with 5 folds.


In [24]:
cv_results = []

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

for model_name, pipeline in models.items():
    print(f"Cross-validating {model_name}...")

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring
    )

    cv_results.append({
        "Model": model_name,
        "Mean MAE": -scores["test_MAE"].mean(),
        "Mean RMSE": -scores["test_RMSE"].mean(),
        "Mean R2": scores["test_R2"].mean(),
        "R2 Std": scores["test_R2"].std()
    })

    print(f"{model_name} completed.\n")

Cross-validating Linear Regression...
Linear Regression completed.

Cross-validating Random Forest...
Random Forest completed.

Cross-validating Gradient Boosting...
Gradient Boosting completed.



In [25]:
cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by="Mean R2",
    ascending=False
).reset_index(drop=True)

cv_results_df

,Model,Mean MAE,Mean RMSE,Mean R2,R2 Std
0,Gradient Boosting,0.722935,1.405158,0.905019,0.083521
1,Random Forest,0.791765,1.672529,0.852193,0.166399
2,Linear Regression,1.231384,1.869526,0.852037,0.054087


In [26]:
for model_name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="r2"
    )

    print(f"\n{model_name}")

    for fold, score in enumerate(scores["test_score"], start=1):
        print(f"Fold {fold}: {score:.4f}")

    print(
        f"Average: {scores['test_score'].mean():.4f}"
    )


Linear Regression
Fold 1: 0.7528
Fold 2: 0.8681
Fold 3: 0.8429
Fold 4: 0.8884
Fold 5: 0.9079
Average: 0.8520

Random Forest
Fold 1: 0.5343
Fold 2: 0.9453
Fold 3: 0.8383
Fold 4: 0.9658
Fold 5: 0.9772
Average: 0.8522

Gradient Boosting
Fold 1: 0.7631
Fold 2: 0.9521
Fold 3: 0.8571
Fold 4: 0.9734
Fold 5: 0.9794
Average: 0.9050


In [29]:
gradient_boosting_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        )
    ]
)

gradient_boosting_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the

In [30]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

total_combinations = np.prod(
    [len(values) for values in param_grid.values()]
)

print("Total Parameter Combinations:", total_combinations)

Total Parameter Combinations: 243


In [31]:
grid_search = GridSearchCV(
    estimator=gradient_boosting_pipeline,
    param_grid=param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.01, 0.05, ...], 'model__max_depth': [2, 3, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know

In [32]:
print("Best Parameters:")

for parameter, value in grid_search.best_params_.items():
    print(f"{parameter}: {value}")

print(
    "\nBest Cross-Validation R2:",
    round(grid_search.best_score_, 4)
)

Best Parameters:
model__learning_rate: 0.05
model__max_depth: 4
model__min_samples_leaf: 1
model__min_samples_split: 5
model__n_estimators: 300

Best Cross-Validation R2: 0.9197


In [33]:
default_gb_result = cv_results_df[
    cv_results_df["Model"] == "Gradient Boosting"
].iloc[0]

print("Default Gradient Boosting")
print(
    "Mean CV R2:",
    round(default_gb_result["Mean R2"], 4)
)

print("\nTuned Gradient Boosting")
print(
    "Mean CV R2:",
    round(grid_search.best_score_, 4)
)

improvement = (
    grid_search.best_score_
    - default_gb_result["Mean R2"]
)

print(
    "\nR2 Improvement:",
    round(improvement, 4)
)

Default Gradient Boosting
Mean CV R2: 0.905

Tuned Gradient Boosting
Mean CV R2: 0.9197

R2 Improvement: 0.0146


In [34]:
final_model = grid_search.best_estimator_

print("Final Model:")
print(final_model)

Final Model:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Present_Price',
                                                   'Kms_Driven', 'Owner',
                                                   'Car_Age']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Fuel_Type', 'Seller_Type',
                                                   'Transmission'])])),
                ('model',
                 GradientBoostingRegressor(learning_rate=0.05, max_depth=4,
                                           min_samples_split=5,
                                           n_estimators=300,
                                           random_state=42))])


In [35]:
final_preprocessor = final_model.named_steps["preprocessor"]
final_regressor = final_model.named_steps["model"]

print("Preprocessor:")
print(final_preprocessor)

print("\nRegressor:")
print(final_regressor)

Preprocessor:
ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Present_Price', 'Kms_Driven', 'Owner',
                                  'Car_Age']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Fuel_Type', 'Seller_Type', 'Transmission'])])

Regressor:
GradientBoostingRegressor(learning_rate=0.05, max_depth=4, min_samples_split=5,
                          n_estimators=300, random_state=42)


In [36]:
feature_names = final_preprocessor.get_feature_names_out()

print("Total Transformed Features:", len(feature_names))

for index, feature_name in enumerate(feature_names):
    print(index, feature_name)

Total Transformed Features: 11
0 num__Present_Price
1 num__Kms_Driven
2 num__Owner
3 num__Car_Age
4 cat__Fuel_Type_CNG
5 cat__Fuel_Type_Diesel
6 cat__Fuel_Type_Petrol
7 cat__Seller_Type_Dealer
8 cat__Seller_Type_Individual
9 cat__Transmission_Automatic
10 cat__Transmission_Manual


In [37]:
print("Number of Estimators:", final_regressor.n_estimators_)
print("Learning Rate:", final_regressor.learning_rate)
print("Estimator Shape:", final_regressor.estimators_.shape)
print("Initial Prediction:", final_regressor.init_.constant_)

Number of Estimators: 300
Learning Rate: 0.05
Estimator Shape: (300, 1)
Initial Prediction: [[4.58963211]]


In [38]:
sample_data = X.head(5)

sample_predictions = final_model.predict(sample_data)

verification_df = sample_data.copy()
verification_df["Actual_Price"] = y.head(5).values
verification_df["Predicted_Price"] = sample_predictions

verification_df

,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Car_Age,Actual_Price,Predicted_Price
0,5.59,27000,Petrol,Dealer,Manual,0,5,3.35,3.526056
1,9.54,43000,Diesel,Dealer,Manual,0,6,4.75,4.945480
2,9.85,6900,Petrol,Dealer,Manual,0,2,7.25,7.553849
3,4.15,5200,Petrol,Dealer,Manual,0,8,2.85,2.775333
4,6.87,42450,Diesel,Dealer,Manual,0,5,4.60,4.539634


In [40]:
fitted_scaler = final_preprocessor.named_transformers_["num"]
fitted_encoder = final_preprocessor.named_transformers_["cat"]

print("Scaler Mean:")
print(fitted_scaler.mean_)

print("\nScaler Scale:")
print(fitted_scaler.scale_)

print("\nEncoder Categories:")

for feature, categories in zip(
    categorical_features,
    fitted_encoder.categories_
):
    print(feature, ":", categories.tolist())

Scaler Mean:
[7.54103679e+00 3.69167525e+04 4.34782609e-02 5.38461538e+00]

Scaler Scale:
[8.55354717e+00 3.89498730e+04 2.48303829e-01 2.89201948e+00]

Encoder Categories:
Fuel_Type : ['CNG', 'Diesel', 'Petrol']
Seller_Type : ['Dealer', 'Individual']
Transmission : ['Automatic', 'Manual']


In [41]:
def export_tree(tree_regressor):
    tree = tree_regressor.tree_

    return {
        "children_left": tree.children_left.tolist(),
        "children_right": tree.children_right.tolist(),
        "feature": tree.feature.tolist(),
        "threshold": tree.threshold.tolist(),
        "value": tree.value[:, 0, 0].tolist()
    }

In [42]:
browser_model = {
    "model_type": "GradientBoostingRegressor",

    "feature_names": feature_names.tolist(),

    "numerical_features": numerical_features,

    "categorical_features": categorical_features,

    "scaler": {
        "mean": fitted_scaler.mean_.tolist(),
        "scale": fitted_scaler.scale_.tolist()
    },

    "encoder": {
        feature: categories.tolist()
        for feature, categories in zip(
            categorical_features,
            fitted_encoder.categories_
        )
    },

    "gradient_boosting": {
        "initial_prediction": float(
            final_regressor.init_.constant_[0, 0]
        ),
        "learning_rate": float(
            final_regressor.learning_rate
        ),
        "n_estimators": int(
            final_regressor.n_estimators_
        ),
        "trees": [
            export_tree(estimator[0])
            for estimator in final_regressor.estimators_
        ]
    }
}

print("Browser model created.")

print(
    "Trees exported:",
    len(browser_model["gradient_boosting"]["trees"])
)

Browser model created.
Trees exported: 300


In [43]:
output_directory = Path("../public/models")

output_directory.mkdir(
    parents=True,
    exist_ok=True
)

output_path = output_directory / "model.json"

with open(
    output_path,
    "w",
    encoding="utf-8"
) as model_file:
    json.dump(
        browser_model,
        model_file,
        separators=(",", ":")
    )

print("Model exported to:")
print(output_path.resolve())

print(
    "Model file size:",
    round(output_path.stat().st_size / 1024, 2),
    "KB"
)

Model exported to:
D:\car-market-value-predictor\public\models\model.json
Model file size: 281.93 KB


In [44]:
with open(
    output_path,
    "r",
    encoding="utf-8"
) as model_file:
    loaded_browser_model = json.load(model_file)

print(
    "Model Type:",
    loaded_browser_model["model_type"]
)

print(
    "Number of Features:",
    len(loaded_browser_model["feature_names"])
)

print(
    "Number of Trees:",
    len(
        loaded_browser_model[
            "gradient_boosting"
        ]["trees"]
    )
)

print(
    "Learning Rate:",
    loaded_browser_model[
        "gradient_boosting"
    ]["learning_rate"]
)

Model Type: GradientBoostingRegressor
Number of Features: 11
Number of Trees: 300
Learning Rate: 0.05


In [45]:
def predict_tree(tree, features):
    node = 0

    while tree["children_left"][node] != -1:
        feature_index = tree["feature"][node]
        threshold = tree["threshold"][node]

        if features[feature_index] <= threshold:
            node = tree["children_left"][node]
        else:
            node = tree["children_right"][node]

    return tree["value"][node]


def preprocess_input(row, model_data):
    numerical_values = [
        row[feature]
        for feature in model_data["numerical_features"]
    ]

    scaled_numerical_values = [
        (value - mean) / scale
        for value, mean, scale in zip(
            numerical_values,
            model_data["scaler"]["mean"],
            model_data["scaler"]["scale"]
        )
    ]

    encoded_categorical_values = []

    for feature in model_data["categorical_features"]:
        categories = model_data["encoder"][feature]
        selected_value = row[feature]

        encoded_categorical_values.extend([
            1.0 if selected_value == category else 0.0
            for category in categories
        ])

    return (
        scaled_numerical_values
        + encoded_categorical_values
    )


def predict_from_json(row, model_data):
    features = preprocess_input(
        row,
        model_data
    )

    gradient_boosting = model_data[
        "gradient_boosting"
    ]

    prediction = gradient_boosting[
        "initial_prediction"
    ]

    for tree in gradient_boosting["trees"]:
        tree_prediction = predict_tree(
            tree,
            features
        )

        prediction += (
            gradient_boosting["learning_rate"]
            * tree_prediction
        )

    return prediction

In [46]:
print("Python Model vs JSON Model\n")

for index in range(5):
    row = X.iloc[index]

    sklearn_prediction = final_model.predict(
        X.iloc[[index]]
    )[0]

    json_prediction = predict_from_json(
        row,
        loaded_browser_model
    )

    difference = abs(
        sklearn_prediction
        - json_prediction
    )

    print(f"Record {index + 1}")
    print(
        "Scikit-learn Prediction:",
        sklearn_prediction
    )
    print(
        "JSON Prediction:",
        json_prediction
    )
    print(
        "Difference:",
        difference
    )
    print()

Python Model vs JSON Model

Record 1
Scikit-learn Prediction: 3.5260558793634935
JSON Prediction: 3.5260558793634935
Difference: 0.0

Record 2
Scikit-learn Prediction: 4.9454801074107175
JSON Prediction: 4.9454801074107175
Difference: 0.0

Record 3
Scikit-learn Prediction: 7.553849465581028
JSON Prediction: 7.553849465581028
Difference: 0.0

Record 4
Scikit-learn Prediction: 2.7753326984330613
JSON Prediction: 2.7753326984330613
Difference: 0.0

Record 5
Scikit-learn Prediction: 4.53963380110673
JSON Prediction: 4.53963380110673
Difference: 0.0

